# Intra-City Learning module

## Tasks to be done:

1. ~~Add the temporal part (dynamic metadata) for certain timebins we can run it on Jakarta~~
2. ~~Add the minimum speed attribute.~~
3. Fix the Average Speed and Min Speed Attribute prediction as they output nan


In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [ ]:
# Imports 
import numpy as np
import pandas as pd
from pympler import asizeof
from DBHandler import DBHandler

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv

import os

In [ ]:
# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = 'cpu'
print("Device:", device)

print(torch.cuda.is_available())
print(torch.__version__)
print(torch.version.cuda)
print(torch.backends.cuda.is_built())

In [ ]:
import json
with open('cities.json', 'r') as f:
    city_bounds = json.load(f)

In [ ]:
city_bounds

In [ ]:
# =========================
# CELL 0) Load graph G from DB
# =========================


# REPO_ROOT = Path(r"C:\Users\Ahmed sameh\Downloads\mapedia-master")
# REPO_ROOT = Path('/home/spatialuser/websites/mapedia')
# sys.path.insert(0, str(REPO_ROOT))
# sys.modules.pop("modules", None)


db_handler = DBHandler()
db_handler.connect_to_db()

city = 'jakarta'
edges = db_handler.get_edges_enriched_df_streaming(
    min_lat= city_bounds[city]['min_lat'],
    max_lat= city_bounds[city]['max_lat'],
    min_lon= city_bounds[city]['min_lon'],
    max_lon= city_bounds[city]['max_lon']
)

# edges = edges[['pgr_id', 'osm_id', 'length', 'source', 'target', 'geometry',
#        'oneway', 'road_type', 'width', 'nlanes', 'max_speed', 'min_speed',
#        'avg_speed']].copy()
# edges.rename(columns={'pgr_id': 'id'}, inplace=True)
# edges.head()

# print(type(edges), len(edges))

# graph_size_bytes = asizeof.asizeof(edges)
# print(f"Memory Size: {graph_size_bytes / 1024**2:.2f} MB")
# print(f"Memory Size: {graph_size_bytes / 1024**3:.2f} GB")

# edges.plot()


In [ ]:
print('Number of OSM ids: ', len(edges.osm_id.unique()))

null_counts = edges.isnull().sum()
availability = (1 - edges.isnull().mean()) * 100

summary = pd.DataFrame({
    "null_count": null_counts,
    "availability": availability
})

print("After merge summary:")
print(summary)
print("Column types:")
print(edges.info())

In [ ]:
edges['avg_speed']

In [ ]:
edges['avg_speed'].to_numpy()

In [ ]:
import numpy as np

# Convert to matrix: shape (425377, 671)
speed_matrix = np.array(
    edges['avg_speed'].tolist(),
    dtype=np.float32
)

print(speed_matrix.shape)

In [ ]:
n_avail_entries = np.sum(~np.isnan(speed_matrix))
n_entries = speed_matrix.shape[0]*speed_matrix.shape[1]
print(n_avail_entries)
print(n_entries)
print('availability: ', n_avail_entries/n_entries * 100)

In [ ]:
speed_matrix_reshaped = speed_matrix.reshape(-1, 4, 7, 24)
speed_matrix_reshaped.shape

In [ ]:
# Original shape: (452533, 4, 7, 24)
# axes:            roads, seasons, days, hours

# ── 1. Remove seasons: average across seasons (axis=1) ──────────────────
# Shape: (452533, 7, 24)
avg_across_seasons = np.nanmean(speed_matrix_reshaped, axis=1)
print('', avg_across_seasons.shape)

# ── 2. Split days into weekdays (0-4) and weekends (5-6) ────────────────
weekdays = avg_across_seasons[:, :5, :]   # (452533, 5, 24)
print('weekdays', weekdays.shape)
weekends = avg_across_seasons[:, 5:, :]   # (452533, 2, 24)
print('weekends', weekends.shape)

# Average each group → shape: (452533, 24)
avg_weekdays = np.nanmean(weekdays, axis=1)
print('weekends', weekends.shape)
avg_weekends = np.nanmean(weekends, axis=1)
print('weekends', weekends.shape)

# ── 3. Split 24 hours into 6 periods of 4 hours each ────────────────────
# Periods: 00-04, 04-08, 08-12, 12-16, 16-20, 20-24
# Shape after reshape: (452533, 6, 4)
weekdays_by_period = avg_weekdays.reshape(-1, 6, 4)   # (452533, 6, 4)
print('weekdays_by_period', weekdays_by_period.shape)
weekends_by_period = avg_weekends.reshape(-1, 6, 4)   # (452533, 6, 4)
print('weekends_by_period', weekends_by_period.shape)

# Average each period → shape: (452533, 6)
avg_weekdays_periods = np.nanmean(weekdays_by_period, axis=2)
print('avg_weekdays_periods', avg_weekdays_periods.shape)
avg_weekends_periods = np.nanmean(weekends_by_period, axis=2)
print('avg_weekends_periods', avg_weekends_periods.shape)

print(avg_weekdays_periods.shape)  # (452533, 6)
print(avg_weekends_periods.shape)  # (452533, 6)

# ── Summary of period labels ─────────────────────────────────────────────
period_labels = ["00-04", "04-08", "08-12", "12-16", "16-20", "20-24"]

# Shape: (452533, 2, 6)  →  [road, weekday/weekend, period]
final = np.stack([avg_weekdays_periods, avg_weekends_periods], axis=1)
print(final.shape)  # (452533, 2, 6)

In [ ]:
# remove the season column average across this dimension ignoring NaN
# split the days into weekends and weekdays and average each
# split the days into periods of six periods and average them

In [ ]:
n_avail_entries = np.sum(~np.isnan(final))
n_entries = final.shape[0]*final.shape[1]*final.shape[2]
print(n_avail_entries)
print(n_entries)
print('availability: ', n_avail_entries/n_entries * 100)

In [ ]:
final.shape

In [ ]:
edges.columns

In [ ]:
keep_cols = ["source","target","pgr_id","osm_id","oneway","road_type","nlanes","width","length","geometry","max_speed","min_speed"]
missing = [c for c in keep_cols if c not in edges.columns]
if missing: raise ValueError(f"Missing expected columns after merge/clean: {missing}")

if edges.crs == "EPSG:4326":
    print("Already in EPSG:4326")
else:
    print("Not in EPSG:4326")

In [ ]:
def nlanes_to_class(s):
  return np.select(
        [s.isna(), s <= 1, s <= 2], #s <= 3, s <= 4, s <= 5],
        [-1, 0, 1],# 2, 3, 4],
        default=2 #5 # all other nlanes 6+ are in one class
    )
class ZScaler:
    """Z-score scaler that ignores NaN."""
    def __init__(self):
        self.mu = None
        self.sd = None

    def fit(self, x: np.ndarray):
        self.mu = np.nanmean(x)
        self.sd = np.nanstd(x) + 1e-8

    def transform(self, x: np.ndarray):
        return (x - self.mu) / self.sd

# additional value adjustment
edges.oneway = edges.oneway.replace({
    2.0: 1.0, # since opposite direction oneway is considered oneway as well
    3.0: pd.NA, # as alternative is not oneway not two-way
})


edges["nlanes_cls"] = nlanes_to_class(edges["nlanes"])
# TODO: need to take into account the classification of other variables like max_speed, oneway, min_speed and how they not having masked values in them
edges["oneway"] = edges["oneway"].astype("Float64")

edges.to_crs(epsg=3857, inplace=True)
        

In [ ]:
edges.head()

In [ ]:
# TODO: Both tokens could have been converted into -1, -2 integers values without having to go through all of this
MASK_TOKEN = "__MASK__"
UNK_TOKEN = "__UNK__"

highway_vals = edges["road_type"].fillna(UNK_TOKEN).astype(str).values
unique_highways = sorted(pd.unique(highway_vals).tolist())

if UNK_TOKEN not in unique_highways:
    unique_highways.append(UNK_TOKEN)
unique_highways.append(MASK_TOKEN)

hwy2id = {h: i for i, h in enumerate(unique_highways)}
id2hwy = {i: h for h, i in hwy2id.items()}

edges["highway_id"] = pd.Series(highway_vals).map(hwy2id)
HIGHWAY_MASK_ID = hwy2id[MASK_TOKEN]

In [ ]:
hwy2id

In [ ]:
edges["highway_id"]

In [ ]:
# =========================
# 4) SPATIAL split — longitude-quantile bands (simple, contiguous, no leakage)
# =========================
N = len(edges)
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.85, 0.05, 0.1
SPLIT_AXIS = "lon"   # "lon" = cut along longitude (x), "lat" = along latitude (y)
SPLIT_AXIS = "lat"   # "lon" = cut along longitude (x), "lat" = along latitude (y)

centroids = edges.geometry.centroid
coord = (centroids.x if SPLIT_AXIS == "lon" else centroids.y).to_numpy()
order = np.argsort(coord, kind="stable")

In [ ]:
order

In [ ]:
final.shape

In [ ]:


n_train = int(round(TRAIN_FRAC * N))
n_val   = int(round(VAL_FRAC   * N))

train_idx = np.sort(order[:n_train]).astype(np.int64)
val_idx   = np.sort(order[n_train : n_train + n_val]).astype(np.int64)
test_idx  = np.sort(order[n_train + n_val :]).astype(np.int64)

print(f"[Spatial split] axis={SPLIT_AXIS}  train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0

In [ ]:
# =========================
# 5) Normalize continuous features (fit train-only)
# =========================
# KEEP length as an INPUT feature, but DO NOT create y_length and DO NOT predict it.

# 1. Flatten to (N, 12) -- 12 features per road (2 day_types x 6 periods)
avg_speed_flat = final.reshape(N, -1).astype(np.float32)  # (N, 12)

length_raw = edges["length"].to_numpy(dtype=np.float32)
length_log = np.log1p(length_raw)

width_raw = edges["width"].to_numpy(dtype=np.float32)  # may contain NaN
max_raw = edges["max_speed"].to_numpy(dtype=np.float32)
min_raw = edges["min_speed"].to_numpy(dtype=np.float32)

avg_scaler = ZScaler()
len_scaler = ZScaler()
wid_scaler = ZScaler()
max_scaler = ZScaler()
min_scaler = ZScaler()

avg_scaler.fit(avg_speed_flat[train_idx])
len_scaler.fit(length_log[train_idx])
wid_scaler.fit(width_raw[train_idx])
max_scaler.fit(max_raw[train_idx])
min_scaler.fit(min_raw[train_idx])

avg_speed_z = avg_scaler.transform(avg_speed_flat).astype(np.float32)
length_z = len_scaler.transform(length_log).astype(np.float32)
width_z  = wid_scaler.transform(width_raw).astype(np.float32)  # NaN stays NaN
max_z = max_scaler.transform(max_raw).astype(np.float32)
min_z = min_scaler.transform(min_raw).astype(np.float32)

avg_speed_missing = np.isnan(avg_speed_flat).astype(np.float32)
width_missing  = np.isnan(width_z).astype(np.float32)
length_missing = np.zeros_like(width_missing, dtype=np.float32)  # length is complete usually
max_missing = np.isnan(max_z).astype(np.float32)
min_missing = np.isnan(min_z).astype(np.float32)

avg_speed_z = np.nan_to_num(avg_speed_z, nan=0.0)
width_z = np.nan_to_num(width_z, nan=0.0)
max_z = np.nan_to_num(max_z, nan=0.0)
min_z = np.nan_to_num(min_z, nan=0.0)

# continuous features for ALL nodes:
# [length_z, width_z, length_missing, width_missing, len_masked_flag, wid_masked_flag]
# ── Stack everything together ────────────────────────────────────────────
x_cont_all = np.column_stack([
    length_z,           # (N,)
    width_z,            # (N,)
    max_z,              # (N,)
    min_z,              # (N,)
    avg_speed_z,        # (N, 12)  -- flattened weekday/weekend x period
    length_missing,     # (N,)
    width_missing,      # (N,)
    max_missing,        # (N,)
    min_missing,        # (N,)
    avg_speed_missing,  # (N, 12)
    np.zeros(N, dtype=np.float32),   # len_mask
    np.zeros(N, dtype=np.float32),   # wid_mask
    np.zeros(N, dtype=np.float32),   # max_mask
    np.zeros(N, dtype=np.float32),   # min_mask
    np.zeros((N, 12), dtype=np.float32),  # avg_speed_mask
]).astype(np.float32)

print(x_cont_all.shape)

In [ ]:
avg_speed_flat = final.reshape(N, -1).astype(np.float32)  # (N, 12)
print(np.sum(np.isnan(avg_speed_flat)))
print(np.sum(np.isnan(final)))

In [ ]:
np.sum(np.isnan(avg_speed_flat))

In [ ]:
4295279/(425377*12)

In [ ]:
avg_speed_flat.shape

In [ ]:
x_cont_all.shape

In [ ]:
# =========================
# 2) Build line graph adjacency from (u,v)
# =========================
def build_line_graph_edge_index(df, u_col="u", v_col="v", eid_col="eid"):
    """
    Nodes in line graph = edges in original graph (df rows).
    Two line-graph nodes connect if original edges share an endpoint (u or v).
    """
    incident = {}
    for u, v, eid in zip(df[u_col].values, df[v_col].values, df[eid_col].values):
        incident.setdefault(u, []).append(eid)
        if v != u:
            incident.setdefault(v, []).append(eid)

    pairs = set()
    for lst in incident.values():
        for i in range(len(lst)):
            for j in range(i + 1, len(lst)):
                a, b = lst[i], lst[j]
                pairs.add((a, b))
                pairs.add((b, a))

    if not pairs:
        return torch.empty((2, 0), dtype=torch.long)

    pairs_tensor = torch.tensor(list(pairs), dtype=torch.long)
    return pairs_tensor.t().contiguous()

In [ ]:
edges.pgr_id.nunique()

In [ ]:
len(edges.pgr_id)

In [ ]:
edges

In [ ]:
edges = edges.reset_index().rename(columns={'index':'idx'})

In [ ]:
edges.source.max()

In [ ]:
test_idx

In [ ]:
# =========================
# 6) Build induced subgraph per split (spatial-inductive)
# =========================
# Masking nlanes
LANES_MASK_ID = 3   # nlanes classes: 0,1,2 + MASK=3 + MISSING=4
LANES_MISS_ID = 4

# IMPORTANT CHANGE vs your original:
# oneway embedding now supports a MISSING token too:
# 0,1 + MASK=2 + MISSING=3
ONEWAY_MASK_ID = 2
ONEWAY_MISS_ID = 3

edge_index_full = build_line_graph_edge_index(edges, u_col="source", v_col="target", eid_col="idx")

ei = edge_index_full
pairs = ei[0].cpu().numpy().astype(np.int64) * (ei.max().item() + 1) + ei[1].cpu().numpy().astype(np.int64)
dup = len(pairs) - len(np.unique(pairs))
print("duplicate directed edges in line-graph edge_index:", dup)

edge_index_full_np = edge_index_full.cpu().numpy()

# Global targets (NO LENGTH TARGET)
y_avg_speed_all = avg_speed_flat  # (N, 12) -- already float32, NaN for missing
y_highway_all = edges["highway_id"].to_numpy(dtype=np.int64)
y_nlanes_all   = edges["nlanes_cls"].to_numpy(dtype=np.int64)        # -1 for missing
y_oneway_all  = edges["oneway"].to_numpy(dtype=np.float32)       # contains NaN for unknown
y_width_all   = edges["width"].to_numpy(dtype=np.float32)        # may be NaN
y_max_all = edges["max_speed"].to_numpy(dtype=np.float32)
y_min_all = edges["min_speed"].to_numpy(dtype=np.float32)
nlanes_in_all = edges["nlanes_cls"].to_numpy(dtype=np.int64)
nlanes_in_all = np.where(nlanes_in_all == -1, LANES_MISS_ID, nlanes_in_all).astype(np.int64)

# oneway input ids with MISSING token
oneway_in_all = edges["oneway"].to_numpy(dtype=np.float32)  # 0/1/NaN
oneway_in_all = np.where(np.isnan(oneway_in_all), ONEWAY_MISS_ID, oneway_in_all).astype(np.int64)

def build_split_data(split_idx_np):
    split_idx_np = np.array(split_idx_np, dtype=np.int64)
    split_idx_np = np.unique(split_idx_np)
    split_idx_np.sort()
    
    # N must satisfy: N > max(edge_index_full_np)
    # and N > max(split_idx_np)
    assert N > edge_index_full_np.max(), "N too small for edge_index"
    assert N > split_idx_np.max(),       "N too small for split indices"

    map_arr = np.full(N, -1, dtype=np.int64)
    map_arr[split_idx_np] = np.arange(len(split_idx_np), dtype=np.int64)

    src_old = edge_index_full_np[0]
    dst_old = edge_index_full_np[1]

    keep = (map_arr[src_old] >= 0) & (map_arr[dst_old] >= 0)
    src_new = map_arr[src_old[keep]]
    dst_new = map_arr[dst_old[keep]]

    edge_index = torch.from_numpy(np.stack([src_new, dst_new], axis=0)).long()

    data = Data(
        x_cont=torch.from_numpy(x_cont_all[split_idx_np]).float(),
        edge_index=edge_index
    )
    data.num_nodes = len(split_idx_np)
    data.x = data.x_cont

    # inputs
    data.highway_in = torch.from_numpy(y_highway_all[split_idx_np]).long()
    data.nlanes_in   = torch.from_numpy(nlanes_in_all[split_idx_np]).long()
    data.oneway_in  = torch.from_numpy(oneway_in_all[split_idx_np]).long()

    # targets (NO y_length)
    data.y_highway = torch.from_numpy(y_highway_all[split_idx_np]).long()
    data.y_nlanes   = torch.from_numpy(y_nlanes_all[split_idx_np]).long()
    data.y_oneway  = torch.from_numpy(y_oneway_all[split_idx_np]).float()
    data.y_width   = torch.from_numpy(y_width_all[split_idx_np]).float()
    data.y_max = torch.from_numpy(y_max_all[split_idx_np]).float()
    data.y_min = torch.from_numpy(y_min_all[split_idx_np]).float()
    data.y_avg_speed = torch.from_numpy(y_avg_speed_all[split_idx_np]).float()  # (n_split, 12)
    
    print("num_nodes:", data.num_nodes)
    print("edge_index max:", data.edge_index.max().item())
    print("edge_index min:", data.edge_index.min().item())
    print("x shape:", data.x.shape)
    print("y_highway shape:", data.y_highway.shape)
    print("y_nlanes shape:", data.y_nlanes.shape)
    print("y_oneway shape:", data.y_oneway.shape)
    print("y_width shape:", data.y_width.shape)
    print("y_max shape:", data.y_max.shape)
    print("y_min shape:", data.y_min.shape)
    print("y_avg_speed shape:", data.y_avg_speed.shape)

    # Critical check: edge_index must not reference nodes outside [0, num_nodes)
    assert data.edge_index.max().item() < data.num_nodes, \
        f"edge_index references node {data.edge_index.max().item()} but num_nodes={data.num_nodes}"
    assert data.edge_index.min().item() >= 0, "negative node index in edge_index" 
    
    data.validate(raise_on_error=True)
    return data.to(device)

# def avg_speed_loss(pred, target):
#     """pred: (B, 12), target: (B, 12) with NaN for missing"""
#     mask = ~torch.isnan(target)
#     if mask.sum() == 0:
#         return torch.tensor(0.0, device=pred.device)
#     return F.mse_loss(pred[mask], target[mask])

data_train = build_split_data(train_idx)
data_val   = build_split_data(val_idx)
data_test  = build_split_data(test_idx)

print("Train graph:", data_train.num_nodes, "nodes |", data_train.edge_index.shape[1], "edges")
print("Val graph:  ", data_val.num_nodes, "nodes |", data_val.edge_index.shape[1], "edges")
print("Test graph: ", data_test.num_nodes, "nodes |", data_test.edge_index.shape[1], "edges")

def degree_stats(data):
    deg = torch.bincount(data.edge_index[0], minlength=data.num_nodes)
    return {
        "num_nodes": int(data.num_nodes),
        "isolated": int((deg == 0).sum().item()),
        "deg1": int((deg == 1).sum().item()),
        "mean_deg": float(deg.float().mean().item()),
    }

print("Degree stats:")
print("  Train:", degree_stats(data_train))
print("  Val:  ", degree_stats(data_val))
print("  Test: ", degree_stats(data_test))

print('Availability of avg_speed:')
print('train non nans: ', np.sum(~np.isnan(data_train.y_avg_speed.cpu().numpy())))
print('val non nans: ', np.sum(~np.isnan(data_val.y_avg_speed.cpu().numpy())))
print('test non nans: ', np.sum(~np.isnan(data_test.y_avg_speed.cpu().numpy())))

print('Availability of min_speed:')
print('train non nans: ', np.sum(~np.isnan(data_train.y_min.cpu().numpy())))
print('val non nans: ', np.sum(~np.isnan(data_val.y_min.cpu().numpy())))
print('test non nans: ', np.sum(~np.isnan(data_test.y_min.cpu().numpy())))

In [ ]:
edge_index_full_np

In [ ]:
data_train.edge_index

In [ ]:
edges

In [ ]:
print('Availability of avg_speed:')
print('train non nans: ', np.sum(~np.isnan(data_train.y_avg_speed.cpu().numpy())))
print('val non nans: ', np.sum(~np.isnan(data_val.y_avg_speed.cpu().numpy())))
print('test non nans: ', np.sum(~np.isnan(data_test.y_avg_speed.cpu().numpy())))

print('Availability of min_speed:')
print('train non nans: ', np.sum(~np.isnan(data_train.y_min.cpu().numpy())))
print('val non nans: ', np.sum(~np.isnan(data_val.y_min.cpu().numpy())))
print('test non nans: ', np.sum(~np.isnan(data_test.y_min.cpu().numpy())))

In [ ]:
np.sum(~np.isnan(y_min_all))

In [ ]:
# =========================
# 7) Model: embeddings + GATv2 + heads + learnable loss weights
# =========================
class MultiAttrGAT(nn.Module):
    def __init__(self, num_highway, hwy_emb_dim=16,
                 nlanes_emb_dim=8, oneway_emb_dim=4,
                cont_dim=48, avg_speed_dim=12, 
                 hidden=32, heads=2, dropout=0.1):
        super().__init__()

        self.hwy_emb = nn.Embedding(num_highway, hwy_emb_dim)
        self.nlanes_emb = nn.Embedding(5, nlanes_emb_dim)    # 0,1,2, MASK=3, MISS=4
        self.oneway_emb = nn.Embedding(4, oneway_emb_dim)  # 0,1, MASK=2, MISS=3 

        in_dim = cont_dim + hwy_emb_dim + nlanes_emb_dim + oneway_emb_dim

        self.gat1 = GATv2Conv(in_dim, hidden, heads=heads, concat=True, dropout=dropout)
        self.gat2 = GATv2Conv(hidden * heads, hidden, heads=heads, concat=False, dropout=dropout)

        self.head_highway = nn.Linear(hidden, num_highway)
        self.head_nlanes   = nn.Linear(hidden, 3)
        self.head_oneway  = nn.Linear(hidden, 1)
        self.head_width   = nn.Linear(hidden, 1)
        self.head_max = nn.Linear(hidden, 1)
        self.head_min = nn.Linear(hidden, 1)
        self.head_avg_speed = nn.Linear(hidden, avg_speed_dim)  # (B, 12)
        
        # total = Σ exp(-s_i)*L_i + s_i; order: [highway, nlanes, oneway, width, max, min]
        self.log_vars = nn.Parameter(torch.zeros(7))

    def forward(self, x_cont, highway_in, nlanes_in, oneway_in, edge_index):
        hwy = self.hwy_emb(highway_in)
        lan = self.nlanes_emb(nlanes_in)
        onw = self.oneway_emb(oneway_in)

        x = torch.cat([x_cont, hwy, lan, onw], dim=1)

        h = F.elu(self.gat1(x, edge_index))
        h = F.elu(self.gat2(h, edge_index))

        return {
            "highway": self.head_highway(h),
            "nlanes": self.head_nlanes(h),
            "oneway": self.head_oneway(h).squeeze(-1),
            "width": self.head_width(h).squeeze(-1),
            "max_speed": self.head_max(h).squeeze(-1),
            "min_speed": self.head_min(h).squeeze(-1),
            "avg_speed": self.head_avg_speed(h),              # (B, 12)
        }

    def weighted_sum(self, losses_dict):
        L = torch.stack([
            losses_dict["hwy"],
            losses_dict["lan"],
            losses_dict["onw"],
            losses_dict["wid"],
            losses_dict["max"],
            losses_dict["min"],
            losses_dict["avg"],
        ])
        precision = torch.exp(-self.log_vars)
        return torch.sum(precision * L + self.log_vars)

num_highway = len(hwy2id)
model = MultiAttrGAT(num_highway=num_highway, cont_dim=48).to(device)

loss_ce    = nn.CrossEntropyLoss()
loss_bce   = nn.BCEWithLogitsLoss()
loss_huber = nn.SmoothL1Loss()
def avg_speed_loss(pred, target):
    """Masked Huber loss — ignores NaN entries in target."""
    mask = ~torch.isnan(target)
    if mask.sum() == 0:
        return torch.tensor(0.0, device=pred.device)
    return F.smooth_l1_loss(pred[mask], target[mask])

In [ ]:
# =========================
# 8) Masking utils (per-graph)
# =========================
# def bernoulli_mask(num_nodes, valid_mask, p=0.3):
#     r = torch.rand(num_nodes, device=valid_mask.device)
#     return (r < p) & valid_mask
# CHANGE TO
def bernoulli_mask(valid_mask, p=0.3):
    r = torch.rand(valid_mask.shape, device=valid_mask.device)
    return (r < p) & valid_mask

CONT_LENGTH_COL   = 0
CONT_WIDTH_COL    = 1
CONT_MAX_COL      = 2
CONT_MIN_COL      = 3
CONT_AVG_START    = 4   # avg_speed_z occupies columns 4-15 (12 slots)
CONT_AVG_END      = 16  # exclusive


CONT_LENMISS_COL  = 4
CONT_WIDMISS_COL  = 5
CONT_MAXMISS_COL  = 6
CONT_MINMISS_COL  = 7
CONT_AVGMISS_START = 20  # avg_speed_missing: columns 20-31
CONT_AVGMISS_END   = 32

CONT_LENMASK_COL  = 8
CONT_WIDMASK_COL  = 9
CONT_MAXMASK_COL  = 10
CONT_MINMASK_COL  = 11
CONT_AVGMASK_START = 36  # avg_speed_mask: columns 36-47
CONT_AVGMASK_END   = 48

def make_fixed_masks(data, p_mask, seed=999):
    gen = torch.Generator(device=data.y_highway.device)
    gen.manual_seed(seed)

    n = data.num_nodes
    valid_hwy = torch.ones(n, dtype=torch.bool, device=data.y_highway.device)
    valid_lan = (data.y_nlanes != -1)

    # oneway: only valid where we know label (not NaN)
    valid_onw = ~torch.isnan(data.y_oneway)

    valid_wid = ~torch.isnan(data.y_width)
    valid_max = ~torch.isnan(data.y_max)
    valid_min = ~torch.isnan(data.y_min)
    # avg_speed: valid where at least one of the 12 slots is not NaN
    valid_avg = ~torch.isnan(data.y_avg_speed)#.all(dim=1)
    
    # def fixed_mask(valid_mask, p):
    #     r = torch.rand(n, generator=gen, device=valid_mask.device)
    #     return (r < p) & valid_mask
    def fixed_mask(valid_mask, p):
        r = torch.rand(valid_mask.shape, generator=gen, device=valid_mask.device)
        return (r < p) & valid_mask

    return {
        "hwy": fixed_mask(valid_hwy, p_mask),
        "lan": fixed_mask(valid_lan, p_mask),
        "onw": fixed_mask(valid_onw, p_mask),
        "wid": fixed_mask(valid_wid, p_mask),
        "max": fixed_mask(valid_max, p_mask),
        "min": fixed_mask(valid_min, p_mask),
        "avg": fixed_mask(valid_avg, p_mask),
        # NOTE: no "len" mask (length is input-only)
    }

In [ ]:
valid_avg = ~torch.isnan(data_train.y_avg_speed)#.all(dim=1)

In [ ]:
data_train.y_avg_speed

In [ ]:
nan_counts = np.sum(np.isnan(data_train.y_avg_speed.cpu().numpy()), axis=1)

plt.figure(figsize=(10, 4))
plt.hist(nan_counts, bins=50)
plt.xlabel("Number of NaN timesteps")
plt.ylabel("Number of edges")
plt.title("Distribution of NaN counts across edges")
plt.show()

In [ ]:
# import matplotlib.pyplot as plt
# nan_counts = np.sum(np.isnan(data_train.y_avg_speed.cpu().numpy()), axis=1)
# nan_counts_sorted = np.sort(nan_counts)  # small to high

# plt.figure(figsize=(12, 4))
# plt.bar(x=range(len(nan_counts_sorted)), height=nan_counts_sorted)
# plt.xlabel("Edge (sorted by NaN count)")
# plt.ylabel("Number of NaN timesteps")
# plt.title("NaN counts per edge (sorted)")
# plt.tight_layout()
# plt.show()

In [ ]:
valid_avg

In [ ]:


# =========================
# 9) Corruption + losses + metrics
# =========================
def corrupt_inputs_with_flags(data, masks):
    x_cont = data.x_cont.clone()
    highway_in = data.highway_in.clone()
    nlanes_in = data.nlanes_in.clone()
    oneway_in = data.oneway_in.clone()

    highway_in[masks["hwy"]] = HIGHWAY_MASK_ID
    nlanes_in[masks["lan"]]   = LANES_MASK_ID
    oneway_in[masks["onw"]]  = ONEWAY_MASK_ID

    # reset flags
    x_cont[:, CONT_LENMASK_COL] = 0.0
    x_cont[:, CONT_WIDMASK_COL] = 0.0
    x_cont[:, CONT_MAXMASK_COL] = 0.0
    x_cont[:, CONT_MINMASK_COL] = 0.0
    x_cont[:, CONT_AVGMASK_START:CONT_AVGMASK_END] = 0.0

    # IMPORTANT: we still can mask LENGTH as an *input corruption* channel to regularize

    # If we want length to be masked sometimes as a denoising input, we do it here:
    # Example: mask length whenever width is masked (keeps code structure). You can comment it out.
    # x_cont[masks["wid"], CONT_LENGTH_COL] = 0.0
    # x_cont[masks["wid"], CONT_LENMASK_COL] = 1.0

    x_cont[masks["wid"], CONT_WIDTH_COL]  = 0.0
    x_cont[masks["wid"], CONT_WIDMASK_COL] = 1.0

    x_cont[masks["max"], CONT_MAX_COL] = 0.0
    x_cont[masks["max"], CONT_MAXMASK_COL] = 1.0

    x_cont[masks["min"], CONT_MIN_COL] = 0.0
    x_cont[masks["min"], CONT_MINMASK_COL] = 1.0
    
    # corrupt avg_speed (zero out all 12 slots + set mask flags)
    # x_cont[masks["avg"], CONT_AVG_START:CONT_AVG_END]     = 0.0
    # x_cont[masks["avg"], CONT_AVGMASK_START:CONT_AVGMASK_END] = 1.0
    x_cont[:, CONT_AVG_START:CONT_AVG_END][masks["avg"]]      = 0.0
    x_cont[:, CONT_AVGMASK_START:CONT_AVGMASK_END][masks["avg"]] = 1.0

    return x_cont, highway_in, nlanes_in, oneway_in

def compute_losses(pred, data, masks):
    losses = {}

    losses["hwy"] = loss_ce(pred["highway"][masks["hwy"]], data.y_highway[masks["hwy"]]) if masks["hwy"].any() \
        else torch.tensor(0.0, device=device)

    losses["lan"] = loss_ce(pred["nlanes"][masks["lan"]], data.y_nlanes[masks["lan"]]) if masks["lan"].any() \
        else torch.tensor(0.0, device=device)

    losses["onw"] = loss_bce(pred["oneway"][masks["onw"]], data.y_oneway[masks["onw"]]) if masks["onw"].any() \
        else torch.tensor(0.0, device=device)

    losses["wid"] = loss_huber(pred["width"][masks["wid"]], data.y_width[masks["wid"]]) if masks["wid"].any() \
        else torch.tensor(0.0, device=device)

    losses["max"] = loss_huber(pred["max_speed"][masks["max"]], data.y_max[masks["max"]]) if masks["max"].any() \
        else torch.tensor(0.0, device=device)
    
    losses["min"] = loss_huber(pred["min_speed"][masks["min"]], data.y_min[masks["min"]]) if masks["min"].any() \
        else torch.tensor(0.0, device=device)
        
    # avg_speed: masked Huber ignoring NaN slots within each road
    # if masks["avg"].any():
    #     pred_avg   = pred["avg_speed"][masks["avg"]]       # (k, 12)
    #     target_avg = data.y_avg_speed[masks["avg"]]        # (k, 12)
    #     slot_mask  = ~torch.isnan(target_avg)              # (k, 12) ignore NaN slots
    #     losses["avg"] = loss_huber(pred_avg[slot_mask], target_avg[slot_mask]) \
    #         if slot_mask.any() else torch.tensor(0.0, device=device)
    # else:
    #     losses["avg"] = torch.tensor(0.0, device=device)
    # CHANGE TO — masks["avg"] is already (n,12), use it directly as slot_mask
    if masks["avg"].any():
        losses["avg"] = loss_huber(
            pred["avg_speed"][masks["avg"]],
            data.y_avg_speed[masks["avg"]]
        )
    else:
        losses["avg"] = torch.tensor(0.0, device=device)
    total = model.weighted_sum(losses)
    return total, losses

def macro_f1_from_preds(y_true, y_pred, num_classes):
    f1s = []
    for c in range(num_classes):
        tp = ((y_pred == c) & (y_true == c)).sum().float()
        fp = ((y_pred == c) & (y_true != c)).sum().float()
        fn = ((y_pred != c) & (y_true == c)).sum().float()

        denom_p = tp + fp
        denom_r = tp + fn
        prec = tp / denom_p if denom_p > 0 else torch.tensor(0.0, device=y_true.device)
        rec  = tp / denom_r if denom_r > 0 else torch.tensor(0.0, device=y_true.device)

        denom_f = prec + rec
        f1 = (2 * prec * rec / denom_f) if denom_f > 0 else torch.tensor(0.0, device=y_true.device)
        f1s.append(f1)

    return float(torch.stack(f1s).mean())

def binary_auroc(y_true, scores):
    y_true = y_true.float()
    scores = scores.float()

    n_pos = (y_true == 1).sum().item()
    n_neg = (y_true == 0).sum().item()
    if n_pos == 0 or n_neg == 0:
        return np.nan

    sorted_scores, order = torch.sort(scores)
    ranks = torch.empty_like(order, dtype=torch.float32)
    ranks[order] = torch.arange(1, len(scores) + 1, device=scores.device, dtype=torch.float32)

    diffs = torch.diff(sorted_scores)
    tie_starts = torch.where(diffs != 0)[0] + 1
    boundaries = torch.cat([
        torch.tensor([0], device=scores.device),
        tie_starts,
        torch.tensor([len(scores)], device=scores.device),
    ])

    for i in range(len(boundaries) - 1):
        a = int(boundaries[i].item())
        b = int(boundaries[i + 1].item())
        if b - a > 1:
            avg = (a + 1 + b) / 2.0
            ranks[order[a:b]] = avg

    sum_ranks_pos = ranks[y_true == 1].sum()
    n_pos_t = torch.tensor(float(n_pos), device=scores.device)
    n_neg_t = torch.tensor(float(n_neg), device=scores.device)

    auroc = (sum_ranks_pos - n_pos_t * (n_pos_t + 1) / 2.0) / (n_pos_t * n_neg_t)
    return float(auroc)

def compute_metrics(pred, data, masks, num_highway_classes):
    out = {}

    if masks["hwy"].any():
        y_true = data.y_highway[masks["hwy"]]
        y_pred = pred["highway"][masks["hwy"]].argmax(dim=1)
        out["hwy_macro_f1"] = macro_f1_from_preds(y_true, y_pred, num_highway_classes)
    else:
        out["hwy_macro_f1"] = np.nan

    if masks["lan"].any():
        y_true = data.y_nlanes[masks["lan"]]
        y_pred = pred["nlanes"][masks["lan"]].argmax(dim=1)
        out["lan_macro_f1"] = macro_f1_from_preds(y_true, y_pred, 3)
    else:
        out["lan_macro_f1"] = np.nan

    if masks["onw"].any():
        out["onw_auroc"] = binary_auroc(data.y_oneway[masks["onw"]], pred["oneway"][masks["onw"]])
    else:
        out["onw_auroc"] = np.nan

    out["wid_mae_m"] = float(torch.mean(torch.abs(pred["width"][masks["wid"]] - data.y_width[masks["wid"]]))) \
        if masks["wid"].any() else np.nan
    out["max_mae"] = float(torch.mean(torch.abs(pred["max_speed"][masks["max"]] - data.y_max[masks["max"]]))) \
        if masks["max"].any() else np.nan   
    out["min_mae"] = float(torch.mean(torch.abs(pred["min_speed"][masks["min"]] - data.y_min[masks["min"]]))) \
        if masks["min"].any() else np.nan
    # NOTE: no length metric (length is input-only)
    
    # avg_speed MAE: only over masked roads and non-NaN slots
    # if masks["avg"].any():
    #     pred_avg   = pred["avg_speed"][masks["avg"]]
    #     target_avg = data.y_avg_speed[masks["avg"]]
    #     slot_mask  = ~torch.isnan(target_avg)
    #     out["avg_mae"] = float(torch.mean(torch.abs(pred_avg[slot_mask] - target_avg[slot_mask]))) \
    #         if slot_mask.any() else np.nan
    # else:
    #     out["avg_mae"] = np.nan
    # CHANGE TO
    if masks["avg"].any():
        out["avg_mae"] = float(torch.mean(torch.abs(
            pred["avg_speed"][masks["avg"]] - data.y_avg_speed[masks["avg"]]
        )))
    else:
        out["avg_mae"] = np.nan
    return out

@torch.no_grad()
def evaluate_with_masks(model, data, masks, num_highway_classes):
    model.eval()
    x_cont, highway_in, nlanes_in, oneway_in = corrupt_inputs_with_flags(data, masks)
    pred = model(x_cont, highway_in, nlanes_in, oneway_in, data.edge_index)
    total, losses = compute_losses(pred, data, masks)
    metrics = compute_metrics(pred, data, masks, num_highway_classes)
    return total.item(), {k: v.item() for k, v in losses.items()}, metrics

@torch.no_grad()
def evaluate_losses_only(model, data, masks):
    model.eval()
    x_cont, highway_in, nlanes_in, oneway_in = corrupt_inputs_with_flags(data, masks)
    pred = model(x_cont, highway_in, nlanes_in, oneway_in, data.edge_index)
    total, losses = compute_losses(pred, data, masks)
    return total.item(), {k: v.item() for k, v in losses.items()}

In [ ]:
# =========================
# 10) Training loop + logging
# =========================
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 10000
P_MASK = 0.30
EVAL_EVERY = 1

val_masks_fixed = make_fixed_masks(data_val, p_mask=P_MASK, seed=999)

history = {
    "epoch": [],
    "train_total": [],
    "val_total": [],
    "train_losses": {k: [] for k in ["hwy", "lan", "onw", "wid", "max", "min", "avg"]},
    "val_losses":   {k: [] for k in ["hwy", "lan", "onw", "wid", "max", "min", "avg"]},

    "metric_epoch": [],
    "train_metrics": {k: [] for k in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae", "avg_mae"]},
    "val_metrics":   {k: [] for k in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae", "avg_mae"]},

    "log_vars": [],
}

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    n = data_train.num_nodes

    valid_hwy = torch.ones(n, dtype=torch.bool, device=device)
    valid_lan = (data_train.y_nlanes != -1)
    valid_onw = ~torch.isnan(data_train.y_oneway)
    valid_wid = ~torch.isnan(data_train.y_width)
    valid_max = ~torch.isnan(data_train.y_max)
    valid_min = ~torch.isnan(data_train.y_min)
    valid_avg = ~torch.isnan(data_train.y_avg_speed)#.all(dim=1)  # + avg

    train_masks = {
        "hwy": bernoulli_mask(valid_hwy, P_MASK),
        "lan": bernoulli_mask(valid_lan, P_MASK),
        "onw": bernoulli_mask(valid_onw, P_MASK),
        "wid": bernoulli_mask(valid_wid, P_MASK),
        "max": bernoulli_mask(valid_max, P_MASK),
        "min": bernoulli_mask(valid_min, P_MASK),
        "avg": bernoulli_mask(valid_avg, P_MASK),
    }

    x_cont, highway_in, nlanes_in, oneway_in = corrupt_inputs_with_flags(data_train, train_masks)
    # print("x shape:", x_cont.shape)
    # print("edge_index shape:", data_train.edge_index.shape)
    # print("edge min:", data_train.edge_index.min().item())
    # print("edge max:", data_train.edge_index.max().item())
    pred = model(x_cont, highway_in, nlanes_in, oneway_in, data_train.edge_index)
    
    # --- Sanity check pred outputs ---
    for key, tensor in pred.items():
        if torch.isnan(tensor).any():
            print(f"[NaN] pred['{key}']")
        if torch.isinf(tensor).any():
            print(f"[Inf] pred['{key}']")
        print(f"pred['{key}'] shape={tensor.shape}, range=[{tensor.min().item():.3f}, {tensor.max().item():.3f}]")

    # --- Sanity check targets (most likely cause) ---
    # Replace these with your actual target attribute names
    for attr in ["highway", "lanes", "oneway"]:   # <-- adjust to your Data attributes
        if hasattr(data_train, attr):
            t = getattr(data_train, attr)
            print(f"data_train.{attr}: min={t.min().item()}, max={t.max().item()}, dtype={t.dtype}")

    total_loss, losses = compute_losses(pred, data_train, train_masks)
    total_loss.backward()
    optimizer.step()

    val_total, val_losses = evaluate_losses_only(model, data_val, val_masks_fixed)

    history["epoch"].append(epoch)
    history["train_total"].append(total_loss.item())
    history["val_total"].append(val_total)

    for k in ["hwy", "lan", "onw", "wid", "max", "min", "avg"]:
        history["train_losses"][k].append(losses[k].item())
        history["val_losses"][k].append(val_losses[k])

    history["log_vars"].append(model.log_vars.detach().cpu().numpy().copy())

    do_metrics = (epoch == 1) or (epoch % EVAL_EVERY == 0)
    if do_metrics:
        train_metrics = compute_metrics(pred, data_train, train_masks, num_highway)
        _, _, val_metrics = evaluate_with_masks(model, data_val, val_masks_fixed, num_highway)

        history["metric_epoch"].append(epoch)
        for k in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae", "avg_mae"]:
            history["train_metrics"][k].append(train_metrics[k])
            history["val_metrics"][k].append(val_metrics[k])

        print(
            f"Epoch {epoch:04d} | total={total_loss.item():.4f} | "
            f"hwy={losses['hwy'].item():.3f}, lan={losses['lan'].item():.3f}, onw={losses['onw'].item():.3f}, "
            f"wid={losses['wid'].item():.3f} max={losses['max'].item():.3f} min={losses['min'].item():.3f}, "
            f"avg={losses["avg"].item():.3f} | "
            f"VAL fixed: hwy_F1={val_metrics['hwy_macro_f1']:.3f}, lan_F1={val_metrics['lan_macro_f1']:.3f}, "
            f"onw_AUROC={val_metrics['onw_auroc']:.3f}, wid_MAE(m)={val_metrics['wid_mae_m']:.3f} "
            f"max_MAE={val_metrics['max_mae']:.3f}, min_MAE={val_metrics['min_mae']:.3f}, "
            f"avg_MAE={val_metrics['avg_mae']:.3f} | "                                             # + avg
            f"log_vars={model.log_vars.detach().cpu().numpy()}"
        )

In [ ]:
train_metrics = compute_metrics(pred, data_train, train_masks, num_highway)
test_masks_fixed = make_fixed_masks(data_test, p_mask=P_MASK, seed=999)
_, _, test_metrics = evaluate_with_masks(model, data_test, test_masks_fixed, num_highway)
_, _, val_metrics = evaluate_with_masks(model, data_val, val_masks_fixed, num_highway)

print(
    f"Epoch {epoch:04d} | total={total_loss.item():.4f} | "
    f"hwy={losses['hwy'].item():.3f}, lan={losses['lan'].item():.3f}, onw={losses['onw'].item():.3f}, "
    f"wid={losses['wid'].item():.3f} max={losses['max'].item():.3f} min={losses['min'].item():.3f} |\n "
    f"VAL fixed: hwy_F1={val_metrics['hwy_macro_f1']:.3f}, lan_F1={val_metrics['lan_macro_f1']:.3f}, "
    f"onw_AUROC={val_metrics['onw_auroc']:.3f}, wid_MAE(m)={val_metrics['wid_mae_m']:.3f} "
    f"max_MAE={val_metrics['max_mae']:.3f}, min_MAE={val_metrics['min_mae']:.3f} |\n "
    f"TEST fixed: hwy_F1={test_metrics['hwy_macro_f1']:.3f}, lan_F1={test_metrics['lan_macro_f1']:.3f}, "
    f"onw_AUROC={test_metrics['onw_auroc']:.3f}, wid_MAE(m)={test_metrics['wid_mae_m']:.3f} "
    f"max_MAE={test_metrics['max_mae']:.3f}, min_MAE={test_metrics['min_mae']:.3f} |\n "
    f"log_vars={model.log_vars.detach().cpu().numpy()}"
        )

In [ ]:
data_train

In [ ]:
print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

In [ ]:
# =========================
# 11) Plots
# =========================
plt.figure()
plt.plot(history["epoch"], history["train_total"], label="Train")
plt.plot(history["epoch"], history["val_total"], label="Val")
plt.xlabel("Epoch")
plt.ylabel("Total Loss")
plt.title("Total Loss (Train vs Val)")
plt.legend()
plt.show()

task_titles = {
    "hwy": "Highway CE",
    "lan": "Lanes CE",
    "onw": "Oneway BCE",
    "wid": "Width Huber",
    "max": "Max Speed Huber",
    "min": "Min Speed Huber",
}
for t in ["hwy", "lan", "onw", "wid", "max", "min"]:
    plt.figure()
    plt.plot(history["epoch"], history["train_losses"][t], label="Train")
    plt.plot(history["epoch"], history["val_losses"][t], label="Val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{task_titles[t]} (masked-only)")
    plt.legend()
    plt.show()

metric_titles = {
    "hwy_macro_f1": "Highway Macro-F1 (masked)",
    "lan_macro_f1": "Lanes Macro-F1 (masked)",
    "onw_auroc": "Oneway AUROC (masked)",
    "wid_mae_m": "Width MAE (m, masked)",
    "max_mae": "Max Speed MAE (masked)",
    "min_mae": "Min Speed MAE (masked)",
}
for m in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae"]:
    plt.figure()
    plt.plot(history["metric_epoch"], history["train_metrics"][m], label="Train")
    plt.plot(history["metric_epoch"], history["val_metrics"][m], label="Val")
    plt.xlabel("Epoch")
    plt.ylabel(m)
    plt.title(metric_titles[m])
    plt.legend()
    plt.show()


In [ ]:

# =========================
# 12) Final test eval (fixed test masks) on test induced graph
# =========================
test_masks_fixed = make_fixed_masks(data_test, p_mask=P_MASK, seed=2025)
test_total, test_losses, test_metrics = evaluate_with_masks(model, data_test, test_masks_fixed, num_highway)
print("TEST fixed-mask metrics:", test_metrics)
print("TEST fixed-mask losses:", test_losses)

In [ ]:
# =========================
# CELL X) Save trained model + preprocessing artifacts (for another city)
# =========================


SAVE_DIR = "./checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

ckpt_path = os.path.join(SAVE_DIR, "nyc_gat_multitask.pt")

checkpoint = {
    # model
    "model_state": model.state_dict(),

    # model config needed to recreate the architecture
    "model_cfg": {
        "num_highway": int(num_highway),
        "hwy_emb_dim": 16,
        "nlanes_emb_dim": 8,
        "oneway_emb_dim": 4,
        "cont_dim": 12,
        "hidden": 32,
        "heads": 2,
        "dropout": 0.1,
    },

    # categorical vocab mapping (CRITICAL for cross-city)
    "hwy2id": hwy2id,
    "id2hwy": id2hwy,

    # special token IDs
    "token_ids": {
        "HIGHWAY_MASK_ID": int(HIGHWAY_MASK_ID),
        "LANES_MASK_ID": int(LANES_MASK_ID),
        "LANES_MISS_ID": int(LANES_MISS_ID),
        "ONEWAY_MASK_ID": int(ONEWAY_MASK_ID),
        "ONEWAY_MISS_ID": int(ONEWAY_MISS_ID),
    },

    # scalers (CRITICAL for consistent normalization)
    # All four are fit on the Jakarta training split and MUST be saved —
    # the model expects z-scored length / width / max_speed / min_speed on input.
    "scalers": {
        "len_mu": float(len_scaler.mu),
        "len_sd": float(len_scaler.sd),
        "wid_mu": float(wid_scaler.mu),
        "wid_sd": float(wid_scaler.sd),
        "max_mu": float(max_scaler.mu),
        "max_sd": float(max_scaler.sd),
        "min_mu": float(min_scaler.mu),
        "min_sd": float(min_scaler.sd),
    },

    # metadata (optional)
    "meta": {
        "seed": int(SEED),
        "split_axis": "lon",
        "p_mask": float(P_MASK),
    }
}

torch.save(checkpoint, ckpt_path)
print(f"Saved checkpoint to: {ckpt_path}")

In [ ]:
# TODO: Remember that we have done scaling of the variables so we need to scale back all values to their original scale before inputing it to the database